In [1]:
import pandas as pd
from recipe_agent import agent

In [80]:
result = await agent.run("Can I substitute almond milk in the pancakes?")
print(result.output)

I don't have a specific pancake recipe to refer to, but generally, almond milk can be substituted for regular milk in most pancake recipes. It may slightly alter the flavor and texture but should still yield a good result. If you're looking for a pancake recipe, please let me know!


In [12]:
result = await agent.run("Can I substitute almond milk in the pancakes?")
print(result.output)

I don't have specific recipes to suggest substitutions for pancake ingredients. However, in general, almond milk can be used as a substitute for regular milk in pancake recipes. It may slightly alter the flavor and texture, but it should work fine. If you need a specific pancake recipe or alternatives, I can help you find some!


In [2]:
df_scenarios = pd.read_json('results.json')

In [3]:
df_scenarios[df_scenarios['question'] == "Can I substitute almond milk in the pancakes?"]

,question,category,type,output,execution_time,tokens,cost,label,comments
5,Can I substitute almond milk in the pancakes?,substitution,ingredient_swap,I don't have a specific pancake recipe to refe...,1.77,"{'input_tokens': 233, 'output_tokens': 56, 'to...",0.000069,bad,Invented substitution tip.


In [22]:
df_results = pd.read_json('results.json')
df_results.label.value_counts()

label
good    14
bad     12
Name: count, dtype: int64

In [23]:
df_results_judged = pd.read_json('results_judged.json')
df_results_judged.judge_label.value_counts()

judge_label
bad     20
good     6
Name: count, dtype: int64

In [6]:
df_results_judged.head()

,question,category,type,output,execution_time,tokens,cost,label,comments,judge_label,judge_reasoning
0,How do I make spaghetti carbonara?,specific-recipe,direct,Here's how to make Spaghetti Carbonara:\n\n###...,8.03,"{'input_tokens': 1036, 'output_tokens': 276, '...",0.000321,good,,bad,The agent response provides a complete recipe ...
1,What can I cook with chicken and coconut milk?,ingredient-search,ingredient-based,Here are two delicious recipes you can cook wi...,12.82,"{'input_tokens': 1571, 'output_tokens': 473, '...",0.000519,good,,bad,The response includes a recipe for Thai Green ...
2,I want something easy and healthy for dinner,preference,vague,Here are three easy and healthy dinner options...,16.75,"{'input_tokens': 1658, 'output_tokens': 535, '...",0.000570,good,,bad,The agent provides three detailed recipes for ...
3,How do I make sushi?,missing-recipe,not-in-collection,I don't have a specific sushi recipe available...,2.24,"{'input_tokens': 492, 'output_tokens': 52, 'to...",0.000105,bad,"The agent suggests alternatives, which is good...",bad,The assistant correctly identified that it doe...
4,What temperature should I set my oven to for t...,recipe-detail,specific,"For the chocolate lava cake, set your oven to ...",3.58,"{'input_tokens': 1035, 'output_tokens': 68, 't...",0.000196,good,,good,The response accurately states that the oven t...


In [24]:
def calculate_metrics(df):
    # True positive (TP): judge says "bad" AND human says "bad"
    tp = ((df.judge_label == 'bad') & (df.label == 'bad')).sum()
    # False positive (FP): judge says "bad" BUT human says "good"
    fp = ((df.judge_label == 'bad') & (df.label == 'good')).sum()
    # False negative (FN): judge says "good" BUT human says "bad"
    fn = ((df.judge_label == 'good') & (df.label == 'bad')).sum()
    # True negative (TN): judge says "good" AND human says "good"
    tn = ((df.judge_label == 'good') & (df.label == 'good')).sum()
    total = len(df)
    
    accuracy = (tp + tn) / total
    precision = tp / (tp + fp)  # when judge says "bad", how often is it right?
    recall = tp / (tp + fn)     # of all actual "bad", how many did the judge catch?
    
    print('accuracy:', accuracy)
    print('precision:', precision)
    print('recall', recall)

In [25]:
calculate_metrics(df_results_judged)

accuracy: 0.6923076923076923
precision: 0.6
recall 1.0


In [26]:
print(
    df_results_judged[
        df_results_judged.question == "Can I substitute almond milk in the pancakes?"
    ].judge_reasoning.iloc[0]
)

The agent incorrectly provides a general substitution suggestion for almond milk in pancakes, which is outside the recipe collection. It should have declined to answer since it does not have a specific pancake recipe to reference and cannot offer advice that isn't directly from the recipe data.


In [27]:
mask_disagreements = (
    (
        (df_results_judged.judge_label == "bad") &
        (df_results_judged.label == "good")
    )
    |
    (
        (df_results_judged.judge_label == "good") &
        (df_results_judged.label == "bad")
    )
)

In [30]:
print(f"number of disagreements before: {len(df_results_judged[mask_disagreements])}")

number of disagreements before: 8


In [29]:
df_results_judged

,question,category,type,output,execution_time,tokens,cost,label,comments,judge_label,judge_reasoning
0,How do I make spaghetti carbonara?,specific-recipe,direct,Here's how to make Spaghetti Carbonara:\n\n###...,8.03,"{'input_tokens': 1036, 'output_tokens': 276, '...",0.000321,good,,bad,The agent response provides a complete recipe ...
1,What can I cook with chicken and coconut milk?,ingredient-search,ingredient-based,Here are two delicious recipes you can cook wi...,12.82,"{'input_tokens': 1571, 'output_tokens': 473, '...",0.000519,good,,bad,The response includes a recipe for Thai Green ...
2,I want something easy and healthy for dinner,preference,vague,Here are three easy and healthy dinner options...,16.75,"{'input_tokens': 1658, 'output_tokens': 535, '...",0.000570,good,,bad,The agent provides three detailed recipes for ...
3,How do I make sushi?,missing-recipe,not-in-collection,I don't have a specific sushi recipe available...,2.24,"{'input_tokens': 492, 'output_tokens': 52, 'to...",0.000105,bad,"The agent suggests alternatives, which is good...",bad,The assistant correctly identified that it doe...
4,What temperature should I set my oven to for t...,recipe-detail,specific,"For the chocolate lava cake, set your oven to ...",3.58,"{'input_tokens': 1035, 'output_tokens': 68, 't...",0.000196,good,,good,The response accurately states that the oven t...
5,Can I substitute almond milk in the pancakes?,substitution,ingredient_swap,I don't have a specific pancake recipe to refe...,1.77,"{'input_tokens': 233, 'output_tokens': 56, 'to...",0.000069,bad,Invented substitution tip.,bad,The agent incorrectly provides a general subst...
6,Can you suggest a vegetarin recipe?,preference,vague,Here are some vegetarian recipes you might enj...,7.16,"{'input_tokens': 688, 'output_tokens': 277, 't...",0.000269,good,,good,The assistant provided three vegetarian recipe...
7,I want pasta,preference,vague,I found a delicious recipe for **Spaghetti Car...,8.36,"{'input_tokens': 1017, 'output_tokens': 311, '...",0.000339,good,,good,The response accurately provides a recipe for ...
8,What Italian dishes do you have?,cuisine-search,direct,Here are some Italian dishes I have:\n\n1. **C...,6.10,"{'input_tokens': 676, 'output_tokens': 261, 't...",0.000258,good,,good,The response accurately lists three Italian di...
9,Can I use bacon instead of guanciale in carbon...,substitution,ingredient-swap,"Yes, you can use bacon as a substitute for gua...",2.00,"{'input_tokens': 237, 'output_tokens': 73, 'to...",0.000079,bad,Makes subsitution tip which isn in the recipe.,bad,The assistant provided cooking advice about su...


In [31]:
def print_bad_label(df_results_judged):
    mask_false_predictions = (
        (
            (df_results_judged.judge_label == "bad") &
            (df_results_judged.label == "good")
        )
        |
        (
            (df_results_judged.judge_label == "good") &
            (df_results_judged.label == "bad")
        )
        |
        (
            (df_results_judged.judge_label == "bad") &
            (df_results_judged.label == "bad")
        )
    )
    
    for question, output, label, comment, judge_label, reasoning in zip(
        df_results_judged[mask_false_predictions].question,
        df_results_judged[mask_false_predictions].output,
        df_results_judged[mask_false_predictions].label,
        df_results_judged[mask_false_predictions].comments,
        df_results_judged[mask_false_predictions].judge_label,
        df_results_judged[mask_false_predictions].judge_reasoning,
    ):
        print(f"Question: {question}")
        print(f"Human label:{label}, judge label:{judge_label}")
        print(f"Human comment:\n{comment}")
        print(f"Judge reasoning:\n{reasoning}")
        print(f"\nOutput: {output}")
        print("-" * 80)

In [85]:
result = await agent.run("How do I make spaghetti carbonara?")
print(result.output)

Here's how to make Spaghetti Carbonara:

### Ingredients:
- Spaghetti
- Guanciale
- Egg yolks
- Pecorino Romano cheese
- Black pepper
- Salt

### Instructions:
1. **Cook the Spaghetti:** Boil water in a pot, add salt, and cook the spaghetti according to the package instructions.
2. **Crisp the Guanciale:** In a pan, cook the guanciale until it becomes crispy.
3. **Prepare the Egg Mixture:** While the pasta is cooking, whisk together the egg yolks with grated pecorino cheese and black pepper in a bowl.
4. **Combine:** Once the spaghetti is cooked, reserve a cup of pasta water and then drain the pasta.
5. **Toss the Pasta:** In the pan with the guanciale, toss in the hot spaghetti. Remove from heat, then quickly stir in the egg mixture. The heat from the pasta will cook the eggs gently.
6. **Make it Creamy:** Add reserved pasta water as needed until you achieve a creamy sauce consistency.
7. **Serve:** Plate the pasta and top with extra pecorino cheese and a sprinkle of black pepper.

##

In [86]:
messages = result.all_messages()
tool_context = []

for msg in messages:
    for p in getattr(msg, "parts", []):

        if p.__class__.__name__ == "ToolReturnPart":
            tool_context.append(p.content)

In [87]:
tool_context

['[6] Spaghetti Carbonara (Italian, medium)\n  Prep: 10min, Cook: 20min, Serves: 4\n  Ingredients: spaghetti, guanciale, egg yolks, pecorino romano, black pepper, salt\n  Tags: classic, pasta\n',
 'Recipe: Spaghetti Carbonara\nCuisine: Italian\nDifficulty: medium\nPrep time: 10 minutes\nCook time: 20 minutes\nServings: 4\n\nIngredients:\n- spaghetti\n- guanciale\n- egg yolks\n- pecorino romano\n- black pepper\n- salt\n\nInstructions:\nCook spaghetti in salted water. Crisp guanciale in a pan. Whisk egg yolks with grated pecorino and black pepper. Drain pasta, reserving some water. Toss hot pasta with guanciale, then quickly stir in egg mixture off heat. Add pasta water as needed for a creamy sauce. Serve immediately with extra pecorino.\n\nTags: classic, pasta']

In [32]:
df_results = pd.read_json('results_20260525_064433.json')
df_results.label.value_counts()

label
good    14
bad     12
Name: count, dtype: int64

In [33]:
df_results_judged = pd.read_json('results_judged_20260525_070722.json')
df_results_judged.judge_label.value_counts()

judge_label
bad     16
good    10
Name: count, dtype: int64

In [34]:
calculate_metrics(df_results_judged)

accuracy: 0.7692307692307693
precision: 0.6875
recall 0.9166666666666666


In [16]:
print_bad_label(df_results_judged)

Question: How do I make sushi?
Human label:bad, judge label:bad
Human comment:
The agent suggests to help find alternatives, which is good. However he cannot  help to find generals guidelines as there are no guidelines in the recipe data. Actually the agent cannot help with that as he is told to answer only based on the recipe data.
Judge reasoning:
The assistant accurately noted that there is no specific recipe for sushi in its collection, which aligns with the recipe data indicating no recipes are available. However, it improperly suggested that alternatives or similar recipes might exist, which could lead the user to think there is additional information available that is not based on the provided recipe collection. This makes the response partially incorrect, falling under guidance beyond the fixed recipe data available.

Output: I'm sorry, but I don't have a specific recipe for sushi in my collection. However, if you're interested in similar recipes or alternatives, please let me 

In [6]:
result = await agent.run("Can I make the falafel with canned chickpeas?")
print(result.output)

The recipe for falafel does not explicitly mention whether canned chickpeas can be used. Therefore, I do not have that information available in the recipe collection. Would you like me to search for similar recipes?


In [35]:
df_results = pd.read_json('results_20260525_085829.json')
df_results.label.value_counts()

label
good    22
bad      4
Name: count, dtype: int64

In [36]:
df_results_judged = pd.read_json('results_judged_20260525_090950.json')
df_results_judged.judge_label.value_counts()

judge_label
good    21
bad      5
Name: count, dtype: int64

In [37]:
calculate_metrics(df_results_judged)

accuracy: 0.8846153846153846
precision: 0.6
recall 0.75


In [38]:
mask_disagreements = (
    (
        (df_results_judged.judge_label == "bad") &
        (df_results_judged.label == "good")
    )
    |
    (
        (df_results_judged.judge_label == "good") &
        (df_results_judged.label == "bad")
    )
)
print(f"number of disagreements after: {len(df_results_judged[mask_disagreements])}")

number of disagreements after: 3


In [39]:
df_results_judged

,question,category,type,tool_context,output,execution_time,tokens,cost,label,comments,judge_label,judge_reasoning
0,How do I make spaghetti carbonara?,specific-recipe,direct,"[[6] Spaghetti Carbonara (Italian, medium)\n ...",Here’s how to make Spaghetti Carbonara:\n\n**I...,6.25,"{'input_tokens': 1546, 'output_tokens': 179, '...",0.000339,good,,good,The response accurately provides the full reci...
1,What can I cook with chicken and coconut milk?,ingredient-search,ingredient-based,"[[7] Thai Green Curry (Thai, medium)\n Prep: ...",You can make **Thai Green Curry** with chicken...,7.38,"{'input_tokens': 1871, 'output_tokens': 239, '...",0.000424,good,,good,The response provides accurate information sou...
2,I want something easy and healthy for dinner,preference,vague,"[[5] Vegetable Stir Fry (Chinese, easy)\n Pre...",Here are three easy and healthy dinner options...,6.63,"{'input_tokens': 1036, 'output_tokens': 257, '...",0.000310,good,,good,The response accurately provided information a...
3,How do I make sushi?,missing-recipe,not-in-collection,[No recipes found matching your search.],I couldn't find any specific recipes for sushi...,1.65,"{'input_tokens': 832, 'output_tokens': 39, 'to...",0.000148,good,,bad,The assistant correctly identifies that there ...
4,What temperature should I set my oven to for t...,recipe-detail,specific,"[[9] Chocolate Lava Cake (French, hard)\n Pre...","For the chocolate lava cake, you should set yo...",3.38,"{'input_tokens': 1545, 'output_tokens': 49, 't...",0.000261,good,,good,The response correctly identifies the temperat...
5,Can I substitute almond milk in the pancakes?,substitution,ingredient_swap,"[[11] Banana Pancakes (American, easy)\n Prep...",The recipe collection does not provide informa...,1.87,"{'input_tokens': 895, 'output_tokens': 42, 'to...",0.000159,good,,good,The response accurately states that the recipe...
6,Can you suggest a vegetarin recipe?,preference,vague,"[[1] Classic Margherita Pizza (Italian, easy)\...",Here are some vegetarian recipes you can try:\...,6.91,"{'input_tokens': 1028, 'output_tokens': 237, '...",0.000296,good,,good,The assistant provided a list of three vegetar...
7,I want pasta,preference,vague,"[[6] Spaghetti Carbonara (Italian, medium)\n ...",Here's a recipe for **Spaghetti Carbonara**:\n...,5.22,"{'input_tokens': 1527, 'output_tokens': 188, '...",0.000342,good,,good,The response accurately provides the recipe fo...
8,What Italian dishes do you have?,cuisine-search,direct,"[[1] Classic Margherita Pizza (Italian, easy)\...",Here are some Italian dishes in the recipe col...,8.52,"{'input_tokens': 1016, 'output_tokens': 263, '...",0.000310,good,,good,The agent accurately provided a list of Italia...
9,Can I use bacon instead of guanciale in carbon...,substitution,ingredient-swap,"[[6] Spaghetti Carbonara (Italian, medium)\n ...",The recipe collection does not contain any inf...,2.56,"{'input_tokens': 900, 'output_tokens': 62, 'to...",0.000172,good,,good,The agent correctly identified that the recipe...
